1. Install Required Libraries

In [ ]:
!pip install pandas scikit-learn transformers datasets torch

2. Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Tobi-Bueck/customer-support-tickets")
print(dataset)


In [ ]:
import pandas as pd

df = pd.DataFrame(dataset['train'])
print(df.head())

3. Data Preprocessing

In [ ]:
# Check columns
print(df.columns)

In [ ]:
df['subject'] = df['subject'].fillna("")
df['body'] = df['body'].fillna("")

In [ ]:
# Encode labels

from sklearn.preprocessing import LabelEncoder

df = df[df['language'] == 'en']
df['text'] = df['subject']
df['queue'] = df['queue'].str.split(" and ").str[0]

# Use queue as label
le = LabelEncoder()
df['label'] = le.fit_transform(df['queue'])


# Optional: check mapping
print(dict(zip(le.classes_, le.transform(le.classes_))))


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)


4. Load Pre-trained Model - BERT

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(le.classes_)
)

5. Tokenization

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example['text'],
        padding="max_length",
        truncation=True,
        max_length=128
    )


In [ ]:
train_dataset = train_dataset.shuffle(seed=42).select(range(2000))
val_dataset = val_dataset.shuffle(seed=42).select(range(200))

In [ ]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)


train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


6. Training Setup

In [ ]:
from transformers import TrainingArguments

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    logging_dir="./logs",
)

7. Evaluation Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

8. Train Model

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

9. Evaluate Model

In [ ]:
trainer.evaluate()

10. Prediction Function

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1)
    pred = probs.argmax().item()

    return le.inverse_transform([pred])[0]

# Example
print(predict("Payment failed twice"))
print(predict("Cannot login to my account"))